In [1]:
import mesa
import numpy as np
import matplotlib.pyplot as plt
from enum import Enum
import json
import time
import sys
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

class AgentType(Enum):
    PASSIVE    = 1
    NORMAL     = 2
    AGGRESSIVE = 3

TYPE_COEFFICIENTS = {
    AgentType.PASSIVE:    0.3,
    AgentType.NORMAL:     0.7,
    AgentType.AGGRESSIVE: 1.0,
}

# ── Plot Style ────────────────────────────────
plt.rcParams.update({
    'font.size':         10,
    'axes.titlesize':    11,
    'axes.labelsize':    10,
    'xtick.labelsize':    9,
    'ytick.labelsize':    9,
    'legend.fontsize':    9,
    'font.family':       'sans-serif',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.grid':         True,
    'grid.alpha':        0.3,
    'grid.linestyle':    '--',
    'figure.dpi':        150,
    'savefig.dpi':       300,
})

print("V12 Random Aggression Experiment: READY")
print(f"Mesa version: {mesa.__version__}")
print()
print("Three scenarios:")
print("  A — Fixed Uniform    (control — V11 approach)")
print("  B — Random Uniform   (same % randomly distributed)")
print("  C — Convergent Random (starts random, self-corrects)")

V12 Random Aggression Experiment: READY
Mesa version: 3.5.1

Three scenarios:
  A — Fixed Uniform    (control — V11 approach)
  B — Random Uniform   (same % randomly distributed)
  C — Convergent Random (starts random, self-corrects)


In [2]:
# ── BASE AGENT ────────────────────────────────
class QueueAgent(mesa.Agent):
    def __init__(self, model, agent_type,
                 aggression_coeff=None):
        super().__init__(model)
        self.agent_type = agent_type

        # Allow override for random assignment
        if aggression_coeff is not None:
            self.type_coeff = aggression_coeff
        else:
            self.type_coeff = (
                TYPE_COEFFICIENTS[agent_type]
            )

        self.urgency = np.random.uniform(0.3, 1.0)
        self.social_inhibition = (
            np.random.uniform(0.2, 0.8)
        )
        self.exited       = False
        self.entry_time   = None
        self.latency      = None

        # Convergence tracking
        self.move_attempts  = 0
        self.move_successes = 0
        self.initial_coeff  = self.type_coeff

    @property
    def behavior_score(self):
        base_score = (self.type_coeff *
                     self.urgency /
                     self.social_inhibition)
        if self.pos:
            neighbors = (
                self.model.grid.get_neighbors(
                    self.pos, moore=True,
                    include_center=False,
                    radius=2
                )
            )
            density = 1.0 + len(neighbors) * 0.05
        else:
            density = 1.0
        return base_score * density

    def step(self):
        if self.exited or self.pos is None:
            return
        if self.entry_time is None:
            self.entry_time = self.model.steps

        x, y = self.pos
        move_prob = min(self.behavior_score, 1.0)
        self.move_attempts += 1

        if np.random.random() < move_prob:
            new_y = y - 1
            if new_y < 0:
                self.model.grid.remove_agent(self)
                self.exited = True
                self.latency = (
                    self.model.steps -
                    self.entry_time
                )
                self.model.exited_count += 1
                self.model.record_agent(self)
                self.model.moves_this_step += 1
                return
            new_pos = (x, new_y)
            contents = (
                self.model.grid
                .get_cell_list_contents([new_pos])
            )
            max_occ = (
                3 if self.type_coeff >= 0.9
                else 2 if self.type_coeff >= 0.6
                else 1
            )
            if len(contents) < max_occ:
                self.model.grid.move_agent(
                    self, new_pos
                )
                self.move_successes += 1
                self.model.moves_this_step += 1

    def update_coeff_from_behavior(self):
        """Convergent random — reclassify based
        on observed movement rate"""
        if self.move_attempts < 5:
            return  # Not enough data yet
        rate = (self.move_successes /
                self.move_attempts)
        if rate > 0.60:
            self.type_coeff = 1.0   # Aggressive
        elif rate < 0.30:
            self.type_coeff = 0.3   # Passive
        else:
            self.type_coeff = 0.7   # Normal


# ── BASE MODEL ────────────────────────────────
class BunchQueueModel(mesa.Model):
    def __init__(self, n_agents=100,
                 width=20, height=30,
                 pct_aggressive=0.85,
                 pct_normal=0.10,
                 mode='fixed',
                 converge_every=20):
        super().__init__()
        self.width        = width
        self.height       = height
        self.steps        = 0
        self.exited_count = 0
        self.total_agents = n_agents
        self.mode         = mode
        self.converge_every = converge_every
        self.moves_this_step = 0

        # Metrics
        self.idle_history  = []
        self.util_history  = []
        self.coeff_history = []  # Track avg coeff
        self.latencies = {
            AgentType.PASSIVE:    [],
            AgentType.NORMAL:     [],
            AgentType.AGGRESSIVE: [],
        }

        self.grid = mesa.space.MultiGrid(
            width, height, torus=False
        )

        n_aggr   = int(n_agents * pct_aggressive)
        n_norm   = int(n_agents * pct_normal)
        n_pass   = n_agents - n_aggr - n_norm

        if mode == 'fixed':
            # ── A: Fixed uniform assignment ───
            agent_types = (
                [AgentType.AGGRESSIVE] * n_aggr +
                [AgentType.NORMAL]     * n_norm +
                [AgentType.PASSIVE]    * n_pass
            )
            np.random.shuffle(agent_types)
            for at in agent_types:
                agent = QueueAgent(self, at)
                self._place(agent)

        elif mode == 'random':
            # ── B: Random uniform assignment ──
            # Same % but coefficients drawn from
            # uniform distribution — no discrete
            # type boundaries
            for _ in range(n_agents):
                r = np.random.random()
                if r < pct_aggressive:
                    coeff = np.random.uniform(
                        0.7, 1.0
                    )
                    at = AgentType.AGGRESSIVE
                elif r < pct_aggressive + pct_normal:
                    coeff = np.random.uniform(
                        0.4, 0.7
                    )
                    at = AgentType.NORMAL
                else:
                    coeff = np.random.uniform(
                        0.3, 0.5
                    )
                    at = AgentType.PASSIVE
                agent = QueueAgent(
                    self, at, aggression_coeff=coeff
                )
                self._place(agent)

        elif mode == 'convergent':
            # ── C: Convergent random ──────────
            # All agents start with random coeff
            # Scheduler reclassifies every N steps
            for _ in range(n_agents):
                coeff = np.random.uniform(0.3, 1.0)
                # Assign nominal type based on coeff
                if coeff >= 0.7:
                    at = AgentType.AGGRESSIVE
                elif coeff >= 0.5:
                    at = AgentType.NORMAL
                else:
                    at = AgentType.PASSIVE
                agent = QueueAgent(
                    self, at,
                    aggression_coeff=coeff
                )
                self._place(agent)

    def _place(self, agent):
        x = np.random.randint(0, self.width)
        y = np.random.randint(
            self.height // 2, self.height
        )
        self.grid.place_agent(agent, (x, y))

    def record_agent(self, agent):
        if agent.latency is not None:
            self.latencies[
                agent.agent_type
            ].append(agent.latency)

    def step(self):
        self.steps += 1
        self.moves_this_step = 0

        # Convergent mode — reclassify agents
        if (self.mode == 'convergent' and
                self.steps % self.converge_every == 0):
            for agent in self.agents:
                if not agent.exited:
                    agent.update_coeff_from_behavior()

        self.agents.shuffle_do("step")

        # Track metrics
        active = (self.total_agents -
                  self.exited_count)
        if active > 0:
            util = min(
                self.moves_this_step / active, 1.0
            )
            idle = 1.0 - util
        else:
            util = 1.0
            idle = 0.0

        self.idle_history.append(idle)
        self.util_history.append(util)

        # Track avg coefficient
        coeffs = [
            a.type_coeff for a in self.agents
            if not a.exited
        ]
        if coeffs:
            self.coeff_history.append(
                np.mean(coeffs)
            )

    def run(self, max_steps=500):
        for _ in range(max_steps):
            self.step()
            if (self.exited_count >=
                    self.total_agents):
                break
        return self.exited_count

    def get_metrics(self):
        avg_idle = (np.mean(self.idle_history)
                   if self.idle_history else 0)
        avg_util = (np.mean(self.util_history)
                   if self.util_history else 0)
        passive  = (
            np.mean(self.latencies[
                AgentType.PASSIVE
            ]) if self.latencies[
                AgentType.PASSIVE
            ] else 0
        )
        return avg_idle, avg_util, passive


print("Three model modes defined!")
print()
print("Mode A — fixed:      discrete type coefficients")
print("Mode B — random:     continuous random coefficients")
print("Mode C — convergent: random start, self-corrects")
print()
print("Convergence mechanism:")
print("  Every 20 steps — reclassify based on")
print("  observed movement rate")
print("  > 60% move rate  → aggressive (1.0)")
print("  < 30% move rate  → passive (0.3)")
print("  30-60% move rate → normal (0.7)")

Three model modes defined!

Mode A — fixed:      discrete type coefficients
Mode B — random:     continuous random coefficients
Mode C — convergent: random start, self-corrects

Convergence mechanism:
  Every 20 steps — reclassify based on
  observed movement rate
  > 60% move rate  → aggressive (1.0)
  < 30% move rate  → passive (0.3)
  30-60% move rate → normal (0.7)


In [3]:
# ── V12 FOCUSED EXPERIMENT ────────────────────
# 3 scales × 3 aggression levels × 3 modes
# 60 runs each — 1620 total simulations
# Auto-saves after each scale/mode combination

import json, time, sys

class Tee:
    def __init__(self, filename):
        self.file   = open(filename, 'w')
        self.stdout = sys.stdout
    def write(self, text):
        self.file.write(text)
        self.stdout.write(text)
        self.file.flush()
    def flush(self):
        self.file.flush()
        self.stdout.flush()

sys.stdout = Tee('/home/jc/v12_output.txt')

print("V12 RANDOM AGGRESSION EXPERIMENT")
print("="*65)
print("Scales:     100, 1000, 5000 agents")
print("Aggression: 75%, 85%, 90%")
print("Modes:      A=Fixed, B=Random, C=Convergent")
print("Runs:       60 per scenario")
print("Total:      1620 simulations")
print("="*65)

start_time = time.time()

scale_configs = [
    {'name': '100',  'n': 100,  'w': 20,
     'h': 30,  'steps': 500},
    {'name': '1000', 'n': 1000, 'w': 65,
     'h': 90,  'steps': 1500},
    {'name': '5000', 'n': 5000, 'w': 140,
     'h': 200, 'steps': 3000},
]

agg_scenarios = [
    {'pct_agg': 0.75, 'pct_norm': 0.15,
     'name': '75%'},
    {'pct_agg': 0.85, 'pct_norm': 0.10,
     'name': '85%'},
    {'pct_agg': 0.90, 'pct_norm': 0.05,
     'name': '90%'},
]

modes = [
    {'mode': 'fixed',      'label': 'A-Fixed'},
    {'mode': 'random',     'label': 'B-Random'},
    {'mode': 'convergent', 'label': 'C-Convergent'},
]

n_runs = 60
v12_results = {}

for scale in scale_configs:
    print(f"\nScale: {scale['name']} agents")
    print("-"*65)
    v12_results[scale['name']] = {}

    for agg in agg_scenarios:
        v12_results[scale['name']][agg['name']] = {}
        print(f"\n  Aggression: {agg['name']}")

        for mode in modes:
            passive_lats      = []
            avg_idles         = []
            avg_utils         = []
            run_passive_means = []
            coeff_finals      = []

            for run in range(n_runs):
                model = BunchQueueModel(
                    n_agents=scale['n'],
                    width=scale['w'],
                    height=scale['h'],
                    pct_aggressive=agg['pct_agg'],
                    pct_normal=agg['pct_norm'],
                    mode=mode['mode'],
                    converge_every=20
                )
                model.run(
                    max_steps=scale['steps']
                )
                avg_idle, avg_util, _ = (
                    model.get_metrics()
                )
                avg_idles.append(avg_idle)
                avg_utils.append(avg_util)

                if model.latencies[
                    AgentType.PASSIVE
                ]:
                    rm = np.mean(
                        model.latencies[
                            AgentType.PASSIVE
                        ]
                    )
                    run_passive_means.append(rm)
                    passive_lats.extend(
                        model.latencies[
                            AgentType.PASSIVE
                        ]
                    )

                # Track final avg coefficient
                # for convergent mode
                if model.coeff_history:
                    coeff_finals.append(
                        model.coeff_history[-1]
                    )

            avg_passive = (
                np.mean(passive_lats)
                if passive_lats else 0
            )
            std_passive = (
                np.std(run_passive_means)
                if run_passive_means else 0
            )
            mean_idle   = np.mean(avg_idles)
            mean_util   = np.mean(avg_utils)
            final_coeff = (
                np.mean(coeff_finals)
                if coeff_finals else 0
            )

            v12_results[
                scale['name']
            ][agg['name']][mode['label']] = {
                'passive':     avg_passive,
                'std':         std_passive,
                'idle':        mean_idle,
                'util':        mean_util,
                'final_coeff': final_coeff,
            }

            elapsed = (
                (time.time() - start_time) / 60
            )
            coeff_str = (
                f"  FinalCoeff:{final_coeff:.2f}"
                if mode['mode'] == 'convergent'
                else ""
            )
            print(
                f"    {mode['label']:<14} "
                f"Passive:{avg_passive:7.1f} "
                f"(±{std_passive:.1f})  "
                f"Idle:{mean_idle*100:5.1f}%  "
                f"Util:{mean_util*100:5.1f}%"
                f"{coeff_str}  "
                f"[{elapsed:.0f}min]"
            )

        # Auto-save after each aggression level
        with open('/home/jc/v12_results.json',
                  'w') as f:
            json.dump(v12_results, f, indent=2)

    elapsed = (time.time() - start_time) / 60
    print(f"\n  Scale {scale['name']} complete "
          f"[{elapsed:.0f}min] — saved")

# ── SUMMARY ───────────────────────────────────
elapsed_total = (
    (time.time() - start_time) / 60
)
print(f"\n{'='*65}")
print(f"V12 COMPLETE — {elapsed_total:.0f} minutes")
print(f"{'='*65}")

print(f"\nPASSIVE LATENCY COMPARISON")
print(f"{'Scale':<8} {'Aggr':<6} "
      f"{'A-Fixed':>12} {'B-Random':>12} "
      f"{'C-Convergent':>14} "
      f"{'B vs A':>8} {'C vs A':>8}")
print("-"*72)

for scale in scale_configs:
    for agg in agg_scenarios:
        r = v12_results[scale['name']][agg['name']]
        a = r['A-Fixed']['passive']
        b = r['B-Random']['passive']
        c = r['C-Convergent']['passive']
        b_diff = ((b - a) / a * 100)
        c_diff = ((c - a) / a * 100)
        b_str = f"{b_diff:+.1f}%"
        c_str = f"{c_diff:+.1f}%"
        print(
            f"{scale['name']:<8} "
            f"{agg['name']:<6} "
            f"{a:>12.1f} "
            f"{b:>12.1f} "
            f"{c:>14.1f} "
            f"{b_str:>8} "
            f"{c_str:>8}"
        )

print(f"\nSTD DEVIATION COMPARISON")
print(f"{'Scale':<8} {'Aggr':<6} "
      f"{'A-Fixed':>12} {'B-Random':>12} "
      f"{'C-Convergent':>14}")
print("-"*55)

for scale in scale_configs:
    for agg in agg_scenarios:
        r = v12_results[scale['name']][agg['name']]
        a = r['A-Fixed']['std']
        b = r['B-Random']['std']
        c = r['C-Convergent']['std']
        print(
            f"{scale['name']:<8} "
            f"{agg['name']:<6} "
            f"{a:>12.1f} "
            f"{b:>12.1f} "
            f"{c:>14.1f}"
        )

print(f"\nCPU IDLE COMPARISON (%)")
print(f"{'Scale':<8} {'Aggr':<6} "
      f"{'A-Fixed':>12} {'B-Random':>12} "
      f"{'C-Convergent':>14}")
print("-"*55)

for scale in scale_configs:
    for agg in agg_scenarios:
        r = v12_results[scale['name']][agg['name']]
        a = r['A-Fixed']['idle'] * 100
        b = r['B-Random']['idle'] * 100
        c = r['C-Convergent']['idle'] * 100
        print(
            f"{scale['name']:<8} "
            f"{agg['name']:<6} "
            f"{a:>11.1f}% "
            f"{b:>11.1f}% "
            f"{c:>13.1f}%"
        )

print(f"\nCONVERGENT MODE — FINAL COEFFICIENT")
print(f"{'Scale':<8} {'Aggr':<6} "
      f"{'Target':>10} {'Achieved':>10} "
      f"{'Converged?':>12}")
print("-"*50)

for scale in scale_configs:
    for agg in agg_scenarios:
        r = v12_results[
            scale['name']
        ][agg['name']]['C-Convergent']
        target   = agg['pct_agg']
        achieved = r['final_coeff']
        # Check if converged toward target
        # Expected final coeff near 0.7-1.0
        # if high aggression assignment working
        converged = (
            "YES" if achieved > 0.65
            else "PARTIAL" if achieved > 0.55
            else "NO"
        )
        print(
            f"{scale['name']:<8} "
            f"{agg['name']:<6} "
            f"{target:>10.2f} "
            f"{achieved:>10.3f} "
            f"{converged:>12}"
        )

print(f"\nFiles saved:")
print(f"  /home/jc/v12_output.txt")
print(f"  /home/jc/v12_results.json")
print(f"\nSCP when complete:")
print(
    f"  scp -P 2222 jc@127.0.0.1:"
    f"/home/jc/v12_output.txt "
    f"C:\\Users\\jcurr\\Desktop\\"
)

V12 RANDOM AGGRESSION EXPERIMENT
Scales:     100, 1000, 5000 agents
Aggression: 75%, 85%, 90%
Modes:      A=Fixed, B=Random, C=Convergent
Runs:       60 per scenario
Total:      1620 simulations

Scale: 100 agents
-----------------------------------------------------------------

  Aggression: 75%
    A-Fixed        Passive:  134.7 (±26.5)  Idle: 57.2%  Util: 42.8%  in]
    B-Random       Passive:  101.8 (±25.1)  Idle: 49.2%  Util: 50.8%  in]
    C-Convergent   Passive:   91.8 (±14.3)  Idle: 68.2%  Util: 31.8%  FinalCoeff:0.30  in]

  Aggression: 85%
    A-Fixed        Passive:  127.8 (±36.3)  Idle: 48.6%  Util: 51.4%  in]
    B-Random       Passive:   95.7 (±23.0)  Idle: 43.1%  Util: 56.9%  in]
    C-Convergent   Passive:   84.6 (±12.9)  Idle: 66.6%  Util: 33.4%  FinalCoeff:0.30  in]

  Aggression: 90%
    A-Fixed        Passive:  127.7 (±37.1)  Idle: 48.9%  Util: 51.1%  in]
    B-Random       Passive:   98.6 (±34.2)  Idle: 42.6%  Util: 57.4%  in]
    C-Convergent   Passive:   86.7 (±